# Concept

At a surface level, PyTorch tensors look and feel just like NumPy arrays. They share similar slicing, shapes, and indexing mechanics. However, PyTorch tensors possess two critical capabilities that standard arrays lack:

    Hardware Acceleration: They can be offloaded directly to parallel computing hardware like GPUs (CUDA) or TPUs to speed up matrix operations.

    The Computational Graph: They can track every mathematical operation performed on them. When you set requires_grad=True, PyTorch constructs a dynamic directed acyclic graph (DAG) in the background. Each tensor points to a grad_fn (gradient function) that records the exact operation that created it.

In [11]:
import torch

# Create a base tensor and enable gradient tracking
x = torch.tensor([3.0], requires_grad=True)
# Perform a simple algebraic operation: y = x^2 + 5
y = x**2 + 5

# Inspect the structural properties
print(f"Tensor Value: {y.item()}")   # Output: 14.0 (3^2 + 5)
print(f"Tracking Gradients? {y.requires_grad}") # Output: True
print(f"Creating Function: {y.grad_fn}") # Output: <PowBackward0 object>

Tensor Value: 14.0
Tracking Gradients? True
Creating Function: <AddBackward0 object at 0x7f41e729b910>


# Exercise

Create a PyTorch tensor named a containing a single float value 2.0. Enable gradient tracking on it. Compute a new tensor b using the formula:
b=a3−4a

Print the resulting value of b and its internal grad_fn property to see which operation is being tracked.

In [12]:
a = torch.tensor([2.0], requires_grad=True)
b =  a**3 -4*a

print(f"Tensor Value: {b.item()}")  
print(f"Creating Function: {b.grad_fn}")

Tensor Value: 0.0
Creating Function: <SubBackward0 object at 0x7f41e76f55a0>


# Concept

Once your forward pass computes an output scalar value (typically a loss value), you must propagate errors backward to compute gradients. Calling .backward() on the final scalar triggers a reverse traversal of the computational graph.

PyTorch uses the Calculus Chain Rule to compute the derivative of the output with respect to each leaf node tensor that has requires_grad=True. These calculated derivatives are accumulated directly into the .grad attribute of the leaf tensors.

In [13]:
x = torch.tensor([3.0], requires_grad=True)
y = x**2 + 5

# Trigger the backward pass calculation
y.backward()

# Access the accumulated gradient stored inside the leaf node x
print(f"Gradient of x (dy/dx): {x.grad.item()}")

Gradient of x (dy/dx): 6.0


# Exercise

Using your previous expression:
b=a3−4a
where a=2.0.

    Analytically calculate the derivative dadb​.

    Write a script that instantiates a=2.0 with gradient tracking, computes b, triggers the backward pass on b, and prints the accumulated gradient value of a (a.grad). Verify that the program's output matches your analytical calculation.

In [ ]:
a.grad.zero_()
b.backward()

print(a.grad)


# Concept

PyTorch graphs are dynamic. By default, the moment you call .backward(), PyTorch completely destroys the internal runtime computational graph memory to optimize execution speeds.

Because of this graph destruction, if you try to call .backward() a second time on the same output node without re-running the forward pass, your code will crash with a runtime error. If your architecture demands multiple backward operations across shared parameters, you must force PyTorch to retain the graph structure in memory.

In [17]:
x = torch.tensor([5.0], requires_grad=True)
y = x * 3

# Pass retain_graph=True to preserve the execution graph memory layout
y.backward(retain_graph=True)
print(f"First backprop check: {x.grad.item()}")   # Output: 3.0

# Now a second backward pass is possible without throwing a runtime crash
y.backward()
# PyTorch ACCUMULATES gradients! It adds the new 3.0 to the existing 3.0
print(f"Second backprop check: {x.grad.item()}")  # Output: 6.0

First backprop check: 3.0
Second backprop check: 6.0


# Exercise

Create a base tensor x = 4.0 with gradient tracking enabled. Define y=2x. Trigger a backward pass on y while explicitly forcing PyTorch to retain the computational graph. Print x.grad. Then, execute a second backward pass on y and print x.grad again to see how gradients accumulate.

In [21]:
x = torch.tensor([4.0], requires_grad=True)
y = 2*x

y.backward(retain_graph=True)
print(x.grad.item())

y.backward(retain_graph=True)
print(x.grad.item())

2.0
4.0


# Concept

During model evaluation or inference (like running test datasets through a fully trained network), we do not update weights. Tracking gradients or building a computational graph during these phases wastes massive amounts of memory and slows down execution speed.

To turn off gradient tracking globally across a block of code, wrap your execution pipeline within the with torch.no_grad(): context manager. Any tensor operations calculated inside this block will run faster and consume significantly less RAM.

In [22]:
x = torch.tensor([4.0], requires_grad=True)

# Normal graph tracking mode
y = x * 2
print(f"Normal Mode - tracking? {y.requires_grad}") # Output: True

# Evaluation/Inference Mode
with torch.no_grad():
    z = x * 2
    print(f"Inference Mode - tracking? {z.requires_grad}") # Output: False

Normal Mode - tracking? True
Inference Mode - tracking? False


# Exercise

Instantiate a weight tensor W containing values [0.5, -1.2] with gradient tracking enabled.

    Multiply this tensor by 2 outside of any context manager and print whether the output tracks gradients.

    Multiply this tensor by 2 inside a torch.no_grad() context manager block and print whether that output tracks gradients.

In [25]:
W = torch.tensor([0.5, -1.2], requires_grad=True)

out1 = W * 2
print(out1.requires_grad)

with torch.no_grad():
    out2 = W * 2

print(out2.requires_grad)

True
False


# Concept

When training a deep learning model, you compute gradients for a batch of data, apply an optimization step to update the model's weights, and then loop to process the next batch.

However, because PyTorch inherently accumulates gradients on leaf nodes (as you saw when using retain_graph), the gradients from your new batch will be added directly to the gradients from your old batch. If you do not explicitly reset these values, your updates will quickly cascade out of control. To prevent this, you must clear out the accumulated derivatives after every single optimization step using .grad.zero_() on your tracking tensors.

In [26]:
w = torch.tensor([2.0], requires_grad=True)

# --- BATCH 1 ---
loss1 = w * 3
loss1.backward()
print(f"Batch 1 Gradient: {w.grad.item()}") # Output: 3.0

# If we do not clear the gradient here, Batch 2 will add onto it!
w.grad.zero_() # Clears the memory back to 0.0 in-place

# --- BATCH 2 ---
loss2 = w * 5
loss2.backward()
print(f"Batch 2 Gradient: {w.grad.item()}") # Output: 5.0 (Clean, not 8.0!)

Batch 1 Gradient: 3.0
Batch 2 Gradient: 5.0


# Exercise

Instantiate a parameter tensor p = 5.0 with gradient tracking enabled.

    Define an operation loss = p * 2. Trigger backpropagation and print the value of p.grad.

    Right below that, define a second operation loss2 = p * 4. Trigger backpropagation on loss2 without clearing the gradient first, and print the value of p.grad to see the accumulated corruption.

    Call the appropriate in-place method to reset the gradient to zero.

    Run loss2.backward() a second time, and print p.grad to verify that it now cleanly shows only the derivative of the second operation.

In [28]:
p = torch.tensor([2.0], requires_grad=True)

loss = p*2
loss.backward()
print(p.grad)

loss2 = p*4
loss2.backward()
print(p.grad)

p.grad.zero_()

loss2 = p * 4
loss2.backward()
print(p.grad)

tensor([2.])
tensor([6.])
tensor([4.])


# FINAL EXERCISE

Objective:
Build a manual single-step gradient descent optimization loop from scratch without using torch.optim. This task mirrors exactly what happens under the hood when a neural network updates its internal weights based on a calculated loss error.

Requirements:
1. Initialization: Create a tracking weight tensor w initialized to 2.0 and a tracking bias tensor b initialized to 1.0. Both must have gradient tracking enabled.
2. Inputs: Define fixed data variables: input feature x=3.0 and ground-truth target label ytrue​=8.0.
3. Forward Pass: Compute a linear prediction using your model variables:
    ypred​=w⋅x+b
4. Loss Calculation: Compute the squared error loss between your prediction and the truth:
    loss=(ypred​−ytrue​)2
5. Backward Pass: Trigger backpropagation on the calculated loss tensor.
6. Manual Gradient Step: Update both variables w and b using the classic Gradient Descent weight update rule:
θ=θ−(η⋅θgrad​). Where θ represents the parameter, θgrad​ is its calculated gradient vector, and η is the learning rate. Use a learning rate of η=0.01.

    Critical Constraint: You must perform this update parameter modification inside a torch.no_grad() context block so PyTorch doesn't attempt to track your optimization adjustments as part of a new computational graph.

    In-place Mutation: Modify the tensors in-place using the subtraction operator with an underscore (-=) on the .data attribute, or assign the values within the context manager directly.
7. Reset: Clear the accumulated gradients on both w and b so they are ready for a clean next iteration cycle.
8. Output Verification: Print the final values of w and b after the single step update has run.

In [30]:
# Parameters
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([1.0], requires_grad=True)

# Data
x = torch.tensor([3.0])
y_true = torch.tensor([8.0])

# Forward pass
y_pred = w * x + b

# Loss
loss = (y_pred - y_true) ** 2

# Backward pass
loss.backward()

# Gradient descent step
lr = 0.01

with torch.no_grad():
    w -= lr * w.grad
    b -= lr * b.grad

# Clear gradients
w.grad.zero_()
b.grad.zero_()

# Verify updated parameters
print("w =", w)
print("b =", b)

w = tensor([2.0600], requires_grad=True)
b = tensor([1.0200], requires_grad=True)
